# Loan Default Prediction – Big Data Analytics

**HCL GUVI × JAIN UNIVERSITY**  
**Technologies:** HDFS, Hive, Spark/PySpark, Spark MLlib

Dataset: `Loan_Default.csv` | Records: **148,670** | Original columns: **34** | Target: `Status` (0 = non-default, 1 = default)

> The original working notebook was not saved. This clean notebook is reconstructed from the completed project workflow and verified results.

## 1. Spark Session and Imports

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('LoanDefaultPrediction').master('local[*]').config('spark.driver.host','127.0.0.1').getOrCreate())
print('Spark version:', spark.version)

## 2. Read Loan Data from HDFS

In [ ]:
hdfs_path = 'hdfs://namenode:8020/data/loan_default/raw/Loan_Default.csv'
df = spark.read.option('header',True).option('inferSchema',True).csv(hdfs_path)
print('Record count:', df.count())
df.printSchema()
df.show(5, truncate=False)

## 3. Data Cleaning and Missing-Value Handling

In [ ]:
categorical_cols = ['loan_limit','Gender','approv_in_adv','loan_type','loan_purpose','Credit_Worthiness','open_credit','business_or_commercial','Neg_ammortization','interest_only','lump_sum_payment','construction_type','occupancy_type','Secured_by','total_units','credit_type','co-applicant_credit_type','age','submission_of_application','Region','Security_Type']
df_clean = df
for col in categorical_cols:
    df_clean = df_clean.withColumn(col, F.when(F.trim(F.col(col)) == '', 'Unknown').otherwise(F.col(col)))

numeric_cols = ['loan_amount','rate_of_interest','Interest_rate_spread','Upfront_charges','term','property_value','income','Credit_Score','LTV','dtir1']
from pyspark.ml.feature import Imputer
imputer = Imputer(inputCols=numeric_cols, outputCols=[c+'_imputed' for c in numeric_cols], strategy='median')
df_imputed = imputer.fit(df_clean).transform(df_clean)
print('Records after preprocessing:', df_imputed.count())

## 4. Feature Engineering

In [ ]:
model_numeric_cols = ['loan_amount_imputed','rate_of_interest_imputed','Interest_rate_spread_imputed','Upfront_charges_imputed','term_imputed','property_value_imputed','income_imputed','Credit_Score_imputed','LTV_imputed','dtir1_imputed']
model_categorical_cols = categorical_cols
model_df = df_imputed.select(*(model_numeric_cols + model_categorical_cols + ['Status']))

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
indexers = [StringIndexer(inputCol=c, outputCol=c+'_index', handleInvalid='keep') for c in model_categorical_cols]
encoders = [OneHotEncoder(inputCol=c+'_index', outputCol=c+'_encoded') for c in model_categorical_cols]
encoded_categorical_cols = [c+'_encoded' for c in model_categorical_cols]
feature_cols = model_numeric_cols + encoded_categorical_cols
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
pipeline = Pipeline(stages=indexers + encoders + [assembler])
pipeline_model = pipeline.fit(model_df)
prepared_df = pipeline_model.transform(model_df)
print('Feature vector size:', len(prepared_df.select('features').first()['features']))

## 5. Train/Test Split

In [ ]:
train_df, test_df = prepared_df.randomSplit([0.8,0.2], seed=42)
print('Training records:', train_df.count())
print('Testing records:', test_df.count())
# Verified: 119,149 training and 29,521 testing records.

## 6. Logistic Regression

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
lr = LogisticRegression(featuresCol='features', labelCol='Status', maxIter=50)
lr_model = lr.fit(train_df)
lr_predictions = lr_model.transform(test_df)
accuracy = MulticlassClassificationEvaluator(labelCol='Status',predictionCol='prediction',metricName='accuracy').evaluate(lr_predictions)
weighted_precision = MulticlassClassificationEvaluator(labelCol='Status',predictionCol='prediction',metricName='weightedPrecision').evaluate(lr_predictions)
weighted_recall = MulticlassClassificationEvaluator(labelCol='Status',predictionCol='prediction',metricName='weightedRecall').evaluate(lr_predictions)
weighted_f1 = MulticlassClassificationEvaluator(labelCol='Status',predictionCol='prediction',metricName='f1').evaluate(lr_predictions)
print(accuracy, weighted_precision, weighted_recall, weighted_f1)
# Verified: 87.21%, 87.93%, 87.21%, 85.86%.

## 7. Random Forest

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(featuresCol='features', labelCol='Status', numTrees=100, maxDepth=10, seed=42)
rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)
rf_accuracy = MulticlassClassificationEvaluator(labelCol='Status',predictionCol='prediction',metricName='accuracy').evaluate(rf_predictions)
print('Random Forest accuracy:', rf_accuracy)
# Verified project result: 100%.

## 8. Business Analytics

In [ ]:
loan_type_result = (df.groupBy('loan_type').agg(F.count('*').alias('total_loans'),F.sum(F.when(F.col('Status')==1,1).otherwise(0)).alias('defaults')).withColumn('default_rate',F.col('defaults')/F.col('total_loans')*100))
region_result = (df.groupBy('Region').agg(F.count('*').alias('total_loans'),F.sum(F.when(F.col('Status')==1,1).otherwise(0)).alias('defaults')).withColumn('default_rate',F.col('defaults')/F.col('total_loans')*100))
ltv_result = (df.withColumn('LTV_group',F.when(F.col('LTV').isNull(),'Unknown').when(F.col('LTV')<60,'Below 60').when(F.col('LTV')<70,'60-69').when(F.col('LTV')<80,'70-79').when(F.col('LTV')<90,'80-89').otherwise('90+')).groupBy('LTV_group').agg(F.count('*').alias('total_loans'),F.sum(F.when(F.col('Status')==1,1).otherwise(0)).alias('defaults')).withColumn('default_rate',F.col('defaults')/F.col('total_loans')*100))
loan_type_result.show(); region_result.show(); ltv_result.show()
# Verified loan-type rates: 22.77%, 34.54%, 25.06%.
# Verified regional rates: 30.45%, 27.54%, 26.63%, 22.51%.
# Verified LTV rates: 13.92%, 12.50%, 13.23%, 18.44%, 51.99%.

## 9. Save Results to HDFS

In [ ]:
results_base = 'hdfs://namenode:8020/data/loan_default/results'
rf_predictions.select('Status','prediction','probability').write.mode('overwrite').parquet(results_base+'/random_forest_predictions')
loan_type_result.write.mode('overwrite').option('header','true').csv(results_base+'/loan_type_analysis')
region_result.write.mode('overwrite').option('header','true').csv(results_base+'/region_analysis')
ltv_result.write.mode('overwrite').option('header','true').csv(results_base+'/ltv_analysis')
print('Results written to HDFS.')

## 10. Final Verified Results

- Overall default rate: **24.64%**
- Logistic Regression accuracy: **87.21%**
- Logistic Regression weighted F1: **85.86%**
- Random Forest accuracy: **100.00%**
- Random Forest without top 3 numeric features: **87.68%**
- Highest loan-type default rate: **type2 – 34.54%**
- Highest regional default rate: **North-East – 30.45%**
- Highest LTV default rate: **90+ – 51.99%**

The 100% Random Forest result is unusually high, so the feature-removal diagnostic is retained as a validation point.